In [0]:
from pyspark.sql.functions import *

In [0]:
base_adls2_path = "abfss://customer360unitycatalog@kmstorage9490.dfs.core.windows.net"
catalog_name = "customer360_cata"
schema_name = "silver"

In [0]:
cust_df = spark.table(f"{catalog_name}.{schema_name}.customers")
ord_df = spark.table(f"{catalog_name}.{schema_name}.orders")
pay_df = spark.table(f"{catalog_name}.{schema_name}.payments")
tickets_df = spark.table(f"{catalog_name}.{schema_name}.support_tickets")
web_df = spark.table(f"{catalog_name}.{schema_name}.web_activities")

In [0]:
customer360_df = cust_df.alias('c').join(ord_df.alias('o'), 'customer_id','left')\
    .join(pay_df.alias('p'), 'customer_id','left')\
    .join(tickets_df.alias('t'), 'customer_id','left')\
    .join(web_df.alias('w'), 'customer_id')\
    .select('c.customer_id','c.name','c.email','c.gender','c.dob','c.city','c.state','c.country','c.zipcode','o.order_id','o.order_date',col('o.amount').alias('order_amount'),col('status').alias('order_status'),'payment_id','payment_date','payment_method','payment_status',col('p.amount').alias('payment_amount'),'ticket_id','issue_type','ticket_date','resolution_status','session_id','page_viewed','session_time','device_type')





In [0]:
customer360_df.write.format("delta").mode("overwrite")\
    .option("overwriteSchema", "true") \
    .option('path',base_adls2_path + '/gold/customer360') \
        .saveAsTable('customer360_cata.gold.customer360')